# 07 - Tydzień 5: Propensity Score Model

Cel tygodnia 5:
- Wytrenować model P(a|s) — klasyfikator 80-klasowy (XGBoost multiclass)
- Sprawdzić kalibrację (reliability diagram)
- Zdiagnozować overlap assumption (ESS, histogram wag, min pscore per action)
- Zastosować model na danych BTS — przygotować `pscores_bts` dla IPS (Tyg. 6)

Kod źródłowy: `src/propensity.py`

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import log_loss, accuracy_score
from obp.dataset import OpenBanditDataset

sys.path.insert(0, str(Path("..").resolve()))
from src.propensity import (
    train_propensity_model,
    extract_propensity_scores,
    effective_sample_size,
    overlap_diagnostics,
)

sns.set_theme(style="whitegrid")
np.random.seed(42)

N_ACTIONS = 80
N_FEATURES = 30
LEN_LIST = 3
RANDOM_STATE = 42
FIGURES_DIR = Path("../figures/week5")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures -> {FIGURES_DIR.resolve()}")

## 1. Wczytanie danych RANDOM policy

In [ ]:
dataset_random = OpenBanditDataset(behavior_policy="random", campaign="all")
feedback_random = dataset_random.obtain_batch_bandit_feedback()

# Używamy danych RANDOM do treningu — mniej selection bias
context = feedback_random["context"][:, :N_FEATURES]   # (10000, 30)
action  = feedback_random["action"]                    # (10000,) integer 0..79
reward  = feedback_random["reward"].astype(np.float32) # (10000,) binary

print(f"Context shape:  {context.shape}")
print(f"Action shape:   {action.shape}  | unique actions: {np.unique(action).shape[0]}")
print(f"Reward mean:    {reward.mean():.4f} (CTR)")
print(f"Action min/max: {action.min()} / {action.max()}")

## 2. Trening modelu propensity scores

In [ ]:
pscore_model = train_propensity_model(
    context=context,
    action=action,
    n_actions=N_ACTIONS,
    random_state=RANDOM_STATE,
)
print(f"Model trained. n_estimators (best): {pscore_model.best_iteration}")
print(f"Classes: {pscore_model.classes_.shape[0]} (expected 80)")

## 3. Diagnostyki modelu

In [ ]:
# Propensity scores na danych treningowych (random policy)
pscores_train = extract_propensity_scores(pscore_model, context, action)

# Metryki klasyfikacyjne (na całym zbiorze — orientacyjne)
proba_all = pscore_model.predict_proba(context)  # (10000, 80)
pred_all  = pscore_model.predict(context)
logloss = log_loss(action, proba_all, labels=np.arange(N_ACTIONS))
acc     = accuracy_score(action, pred_all)

print(f"Multiclass log-loss: {logloss:.4f}")
print(f"Top-1 accuracy:      {acc:.4f}")
print(f"\nPropensity score stats (train):")
print(f"  min={pscores_train.min():.6f}  max={pscores_train.max():.6f}  mean={pscores_train.mean():.4f}")
print(f"  % NaN:         {np.isnan(pscores_train).mean()*100:.2f}%")
print(f"  % below 0.001: {(pscores_train < 0.001).mean()*100:.2f}%")

In [ ]:
# Kalibracja: reliability diagram
# Traktujemy problem binarnie: czy wybrany action = akcja obserwowana?
# Dla każdej obserwacji: y_true = 1, y_prob = P(a_i | s_i)
# Formalnie: sprawdzamy czy model dobrze calibruje szansę trafienia obserwowanej akcji

y_true_binary = np.ones(len(pscores_train))  # zawsze 1 (obserwowaliśmy akcję)
fraction_pos, mean_pred = calibration_curve(
    y_true_binary, pscores_train, n_bins=10, strategy="quantile"
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.plot(mean_pred, fraction_pos, "o-", color="steelblue", label="Model")
ax.set_xlabel("Mean predicted P(a|s)")
ax.set_ylabel("Fraction of observed actions")
ax.set_title("Reliability diagram — Propensity Model Calibration")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "calibration_curve.png", dpi=160)
plt.show()

In [ ]:
# Histogram wag IPS (wstępny overlap check)
pi_eval = 1.0 / N_ACTIONS  # uniform evaluation policy
weights_raw = pi_eval / np.clip(pscores_train, 1e-9, None)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(weights_raw, bins=60, log=True, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(weights_raw.mean(), color="red", linestyle="--", label=f"Mean={weights_raw.mean():.2f}")
ax.axvline(np.percentile(weights_raw, 99), color="orange", linestyle="--", label=f"99th={np.percentile(weights_raw, 99):.2f}")
ax.set_xlabel("IPS weight π_e / P(a|s)")
ax.set_ylabel("Count (log scale)")
ax.set_title("IPS Weight Distribution — unclipped (train split)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ips_weights_hist.png", dpi=160)
plt.show()

In [ ]:
# Effective Sample Size
ess, ess_ratio = effective_sample_size(weights_raw)
print(f"ESS:       {ess:.0f} / {len(weights_raw)} = {ess_ratio:.4f}")
print(f"Próg 0.05: {'OK' if ess_ratio >= 0.05 else 'KRYTYCZNY — duże ryzyko wysokiej wariancji IPS'}")

In [ ]:
# Min pscore per action — overlap heatmap
diag = overlap_diagnostics(pscores_train, action, pi_eval, N_ACTIONS)
min_per_action = diag["min_pscore_per_action"]

fig, ax = plt.subplots(figsize=(14, 2))
bar_colors = ["red" if v < 0.001 else "steelblue" for v in np.nan_to_num(min_per_action)]
ax.bar(np.arange(N_ACTIONS), np.nan_to_num(min_per_action), color=bar_colors, width=0.8)
ax.axhline(0.001, color="red", linestyle="--", linewidth=0.8, label="threshold 0.001")
ax.set_xlabel("Action ID")
ax.set_ylabel("Min P(a|s)")
ax.set_title("Minimum propensity score per action (red = below 0.001 → overlap issue)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "overlap_per_action.png", dpi=160)
plt.show()

n_problem = (np.nan_to_num(min_per_action) < 0.001).sum()
print(f"Akcje z min pscore < 0.001: {n_problem} / {N_ACTIONS}")

## 4. Zastosowanie modelu na danych BTS

In [ ]:
dataset_bts = OpenBanditDataset(behavior_policy="bts", campaign="all")
feedback_bts = dataset_bts.obtain_batch_bandit_feedback()

context_bts = feedback_bts["context"][:, :N_FEATURES]
action_bts  = feedback_bts["action"]

# P(a_bts | s) — używamy modelu wytrenowanego na random policy
pscores_bts = extract_propensity_scores(pscore_model, context_bts, action_bts)

print(f"BTS pscores — shape: {pscores_bts.shape}")
print(f"  min={pscores_bts.min():.6f}  max={pscores_bts.max():.6f}  mean={pscores_bts.mean():.4f}")
print(f"  % NaN:         {np.isnan(pscores_bts).mean()*100:.2f}%")
print(f"  % below 0.001: {(pscores_bts < 0.001).mean()*100:.2f}%")

# Overlap check dla BTS
weights_bts = pi_eval / np.clip(pscores_bts, 1e-9, None)
ess_bts, ess_ratio_bts = effective_sample_size(weights_bts)
print(f"\nESS (BTS): {ess_bts:.0f} / {len(pscores_bts)} = {ess_ratio_bts:.4f}")

## 5. Podsumowanie — Tydzień 5

| Metryka | Wartość |
|---------|--------|
| Multiclass logloss | — |
| ESS ratio (train) | — |
| ESS ratio (BTS) | — |
| Akcje z overlap problem | — / 80 |

**Checklist T5:**
- [ ] `src/propensity.py` — `train_propensity_model`, `extract_propensity_scores`
- [ ] ESS ratio > 0.05
- [ ] Reliability diagram — kalibracja
- [ ] Brak NaN w `pscores_bts`
- [ ] `pscores_bts` gotowe dla Tygodnia 6 (IPS/SNIPS)